# Lab | Agent & Vector store


<br>

## Intro

In this lab you'll build an AI agent that knows when to consult *different* knowledge bases to answer a question — instead of relying on a single source of truth.

Here's what to expect:

1. **Follow a full worked demo** — We'll walk through every step together: ingesting the *state of the union* speech and the *Ruff* docs into two vector stores, wrapping each in a `RetrievalQA` tool, and building an agent that picks the right tool (or both!) depending on the question.

2. **Replicate it yourself with a new dataset** — Then, you'll swap in a dataset of your choice and rebuild the same pipeline, adapting the prompts and tools along the way.

By the end of this lab, you'll understand how to build multi-source AI agents and be able to apply the pattern to your own datasets.

<br>

## Combine agents and vector stores

Let's get into the demo. We'll wrap each vector store in a `RetrievalQA` chain and hand it to an agent as a `Tool`. The agent then decides, at each step, which tool to call based purely on its description — this is what lets it route between multiple knowledge sources.

There are two flavors of this pattern, both of which we'll try below:

- **Agent as reasoner** — the agent calls a tool and can keep reasoning afterward (e.g. to combine results from multiple sources).
- **Agent as router** (`return_direct=True`) — the agent just picks the right tool and returns its answer immediately, no extra reasoning.

<br>

## Install dependencies

Uncomment and run the cells below to install the dependencies required for this notebook.

Tip: Use a virtual environment to keep this project's dependencies isolated from your system Python and other projects.

In [1]:
%pip install -q -U "langchain>=1,<2" "langchain-openai>=1,<2" "langchain-chroma>=1,<2" "langchain-text-splitters>=1,<2" "python-dotenv>=1,<2"

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install -q -U "chromadb>=1,<2" "requests>=2,<3" "beautifulsoup4>=4,<5"

Note: you may need to restart the kernel to use updated packages.


<br>

## Initial Setup

Before building anything, we need to load our API credentials, instantiate the LLM we'll use throughout the notebook, and locate the sample document we'll be querying.

In [3]:
# Modern LangChain imports: no deprecated RetrievalQA / initialize_agent.
from pathlib import Path
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
import requests
from bs4 import BeautifulSoup


In [4]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    from getpass import getpass
    OPENAI_API_KEY = getpass("Enter your OpenAI API key (hidden): ").strip()
if not OPENAI_API_KEY:
    raise RuntimeError("OPENAI_API_KEY is required. Put it in .env or enter it above.")
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
print("API key configured:", bool(OPENAI_API_KEY))


API key configured: True


In [5]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=OPENAI_API_KEY)


<br>

Now let's ingest the state of the union speech: load the raw text, split it into manageable chunks, embed those chunks, and store them in a Chroma vector store.

In [6]:
# Works when the attached text is beside the notebook or under datasets/.
doc_path = next((str(p) for p in [Path("state_of_the_union.txt"), Path("datasets/state_of_the_union.txt")] if p.exists()), None)
if not doc_path:
    raise FileNotFoundError("Place state_of_the_union.txt beside this notebook or in datasets/.")


In [7]:
speech_text = Path(doc_path).read_text(encoding="utf-8")
documents = [Document(page_content=speech_text, metadata={"source":doc_path})]
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
texts = text_splitter.split_documents(documents)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
docsearch = Chroma.from_documents(texts, embeddings, collection_name="state_of_union_lab")
print(f"Indexed {len(texts)} State of the Union chunks")


Indexed 47 State of the Union chunks


<br>

## Adding a second knowledge source

To show how an agent can route between multiple tools, let's add a second vector store — this time built from the Ruff FAQ web page instead of a local file.

In [8]:
def answer_from_store(question, store):
    """Retrieve evidence and answer using the LLM, without deprecated chains."""
    chunks = store.similarity_search(question, k=4)
    context = "\\n\\n".join(d.page_content for d in chunks)
    reply = llm.invoke([
        ("system", "Answer only using the supplied context. If it does not contain the answer, say so. Be concise."),
        ("human", f"Context:\\n{context}\\n\\nQuestion: {question}")
    ])
    return reply.content

def state_of_union(question):
    return answer_from_store(question, docsearch)


In [9]:
# Fetch Ruff documentation without WebBaseLoader compatibility issues.
RUFF_URL = "https://docs.astral.sh/ruff/faq/"


In [10]:
print("Ruff source:", RUFF_URL)

Ruff source: https://docs.astral.sh/ruff/faq/


In [11]:
try:
    response = requests.get(RUFF_URL, timeout=25, headers={"User-Agent": "Mozilla/5.0"})
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    main = soup.find("main") or soup
    for tag in main(["script", "style", "nav", "footer"]):
        tag.decompose()
    ruff_content = main.get_text(" ", strip=True)
    if len(ruff_content) < 300:
        raise ValueError("Ruff page returned insufficient text")
    print("Loaded live Ruff FAQ.")
except (requests.RequestException, ValueError) as exc:
    # Offline fallback, so the rest of the lab still runs.
    ruff_content = """Ruff is a fast Python linter and formatter written in Rust.
Ruff can replace many Flake8 plugins and can also provide formatting functionality.
Ruff supports Jupyter Notebook (.ipynb) files and can lint and format them.
Ruff can be run from the command line, pre-commit hooks and CI pipelines.
Use `ruff check .` to lint Python files and `ruff format .` to format them.
For full details consult https://docs.astral.sh/ruff/faq/"""
    print("Ruff website unavailable; using a small offline fallback:", type(exc).__name__)
ruff_texts = text_splitter.split_documents([Document(page_content=ruff_content, metadata={"source":RUFF_URL})])
ruff_db = Chroma.from_documents(ruff_texts, embeddings, collection_name="ruff_lab")
def ruff(question):
    return answer_from_store(question, ruff_db)
print(f"Indexed {len(ruff_texts)} Ruff FAQ chunks")


Loaded live Ruff FAQ.
Indexed 26 Ruff FAQ chunks


<br>

## Create the Agent

With both `RetrievalQA` chains ready, we wrap each one in a `Tool` (giving it a name and a description the agent will use to decide when to call it), then hand both tools to an agent.

In [12]:
# Modern tool calling: @tool + create_agent (LangChain v1).
@tool
def state_of_union_qa(question: str) -> str:
    """Answer questions about Biden's 2022 State of the Union speech."""
    return state_of_union(question)

@tool
def ruff_qa(question: str) -> str:
    """Answer questions about the Ruff Python linter, formatter and notebooks."""
    return ruff(question)

class LabAgent:
    """Adapter to preserve the original notebook's agent.invoke(...) calls."""
    def __init__(self, toolset, direct=False):
        self.toolset = toolset
        self.direct = direct
        self.inner = create_agent(
            model=llm,
            tools=toolset,
            system_prompt=("Use tools to answer questions based on the documents. "
                           "For questions spanning two sources, call both tools. "
                           "Do not invent details absent from the sources.")
        )
    def invoke(self, query):
        question = query if isinstance(query, str) else query["input"]
        if self.direct:
            # Explicit, deterministic router; no extra agent reasoning after retrieval.
            q = question.lower()
            tool_selected = next((t for t in self.toolset if "ruff" in t.name.lower()), self.toolset[0]) if any(
                word in q for word in ["ruff", "flake8", "linter", "formatter"]
            ) else self.toolset[0]
            return {"output": tool_selected.invoke({"question": question})}
        result = self.inner.invoke({"messages": [{"role": "user", "content": question}]},
                                   config={"recursion_limit": 20})
        return {"output": result["messages"][-1].content, "messages": result["messages"]}


<br>

Let's try it out — first with a question only the state of the union tool can answer, then one only Ruff can answer. Watch the verbose output to see which tool the agent picks each time.

In [13]:
tools = [state_of_union_qa, ruff_qa]
print("Tools:", [t.name for t in tools])

Tools: ['state_of_union_qa', 'ruff_qa']


In [14]:
agent = LabAgent(tools)

In [15]:
agent.invoke(
    "What did biden say about ketanji brown jackson in the state of the union address?"
)

{'output': "In his State of the Union address, Biden described Ketanji Brown Jackson as one of the nation’s top legal minds and a consensus builder. He noted her background as a former top litigator, federal public defender, and her family's history of public service, including educators and police officers. He also highlighted the broad range of support she received since her nomination.",
 'messages': [HumanMessage(content='What did biden say about ketanji brown jackson in the state of the union address?', additional_kwargs={}, response_metadata={}, id='b6e8c507-a35e-4780-a315-857ab3b4f3fa'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 34, 'prompt_tokens': 131, 'total_tokens': 165, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 

In [16]:
agent.invoke("Why use ruff over flake8?")

{'output': 'Ruff can be preferred over Flake8 for several reasons:\n\n1. **More Rules**: Ruff offers over 900 rules, while Flake8 has around 409, providing a broader range of linting options.\n2. **Minimized Conflicts**: Ruff reduces conflicts that can arise from using multiple plugins, which is common with Flake8.\n3. **Easier Plugin Management**: Users can easily toggle individual plugins on or off in Ruff.\n4. **Performance**: Ruff re-implements popular Flake8 plugins in Rust, leading to better performance.\n5. **Simplicity**: Ruff does not support custom lint rules, which may simplify usage for some users.',
 'messages': [HumanMessage(content='Why use ruff over flake8?', additional_kwargs={}, response_metadata={}, id='8acf67e5-310f-40ab-a563-57e1e78809c2'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 121, 'total_tokens': 145, 'completion_tokens_details': {'accepted_prediction_tokens': 0, '

## Use the Agent solely as a router

<br>

You can also set `return_direct=True` if you intend to use the agent as a router and just want to directly return the result of the RetrievalQAChain.

Notice that in the above examples the agent did some extra work after querying the RetrievalQAChain. You can avoid that and just return the result directly.

In [17]:
tools = [state_of_union_qa, ruff_qa]
# The router calls exactly one source and returns its answer directly.

In [18]:
agent = LabAgent(tools, direct=True)

In [19]:
agent.invoke(
    "What did biden say about ketanji brown jackson in the state of the union address?"
)

{'output': 'Biden stated that he nominated Circuit Court of Appeals Judge Ketanji Brown Jackson, describing her as one of the nation’s top legal minds who will continue Justice Breyer’s legacy of excellence. He highlighted her background as a former top litigator, federal public defender, and a consensus builder, noting the broad range of support she has received since her nomination.'}

In [20]:
agent.invoke("Why use ruff over flake8?")

{'output': "Ruff can replace Flake8 and its plugins, offering over 900 rules compared to Flake8's ~409. It minimizes conflicts across plugins, allows toggling of individual plugins easily, and re-implements popular Flake8 plugins in Rust for better performance. Additionally, Ruff does not support custom lint rules, which may simplify usage for some users."}

<br>

## Multi-Hop vector store reasoning

Because vector stores are easily usable as tools in agents, it is easy to use answer multi-hop questions that depend on vector stores using the existing agent framework.

In [21]:
tools = [state_of_union_qa, ruff_qa]

In [22]:
agent = LabAgent(tools)

In [23]:
agent.invoke(
    "What tool does ruff use to run over Jupyter Notebooks? Did the president mention that tool in the state of the union?"
)

{'output': 'Ruff integrates with nbQA to run over Jupyter Notebooks. However, the president did not mention Jupyter Notebooks in the State of the Union.',
 'messages': [HumanMessage(content='What tool does ruff use to run over Jupyter Notebooks? Did the president mention that tool in the state of the union?', additional_kwargs={}, response_metadata={}, id='9ee830db-aa69-4c76-bd14-71a5dafd6a24'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 76, 'prompt_tokens': 139, 'total_tokens': 215, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_73c5e31c8b', 'id': 'chatcmpl-EfYRdg9qrAMf1UlacV3qdZhwHpegv', 'service_tier': 'default', 'finish_reason': 'too

<br><br>

---

<br>

## 🚀 Your turn

Time to make this lab your own! 

Replace the `state_of_the_union.txt` dataset with something you'd actually enjoy chatting with. A great place to start is the [sonnets.txt dataset](https://github.com/martin-gorner/tensorflow-rnn-shakespeare/blob/master/shakespeare/sonnets.txt) —or any other .txt file from that repository. Of course, you're not limited to those options. Pick any text that interests you and see how your chatbot responds.

Here's what to do:

1. **Get your data** — Download your chosen `.txt` file into this project folder (or point `TextLoader` at it directly).
2. **Rebuild the vector store** — Load, split, and embed your new document, then create a fresh `RetrievalQA` chain for it (give it a descriptive `collection_name`!).
3. **Rewrite the tool description** — Update the `Tool`'s `name` and `description` so the agent knows *when* it should reach for this new tool instead of the Ruff or state-of-the-union ones.
4. **Rebuild the agent** — Combine your new tool with the existing Ruff tool (or drop it if you'd rather keep just your new dataset + one other source).
5. **Put it to the test** — Ask your agent:
   - A direct question that only your new dataset can answer.
   - A question that only the Ruff tool can answer.
   - A multi-hop question that requires combining *both* tools' knowledge, like the Jupyter/Ruff example above.
6. **Reflect** — In a markdown cell, briefly note whether the agent picked the right tool(s) each time, and what happened when you set `return_direct=True` vs. not.

⭐️ **Bonus points:**
- Instead of modifying this same file, create a new file `solution.ipynb` and replicate the process from scratch.

💡 **Tip:** 
- Watch the `verbose=True` agent logs closely — they show you the agent's reasoning step by step, which is the best way to understand *why* it picked a particular tool.




### Solution — Custom automotive testing dataset

The original lab text above is preserved. The following cells complete the **Your turn** exercise with a new automotive software testing knowledge base. Put `automotive_testing_knowledge.txt` beside this notebook.


In [25]:
custom_path = Path("datasets/automotive_testing_knowledge.txt")
if not custom_path.exists():
    raise FileNotFoundError("Keep automotive_testing_knowledge.txt beside the notebook")
custom_docs = [Document(page_content=custom_path.read_text(encoding="utf-8"), metadata={"source":str(custom_path)})]
custom_chunks = text_splitter.split_documents(custom_docs)
automotive_db = Chroma.from_documents(custom_chunks, embeddings, collection_name="automotive_testing_lab")
def automotive_qa(question):
    return answer_from_store(question, automotive_db)
print(f"Indexed {len(custom_chunks)} automotive chunks")

FileNotFoundError: Keep automotive_testing_knowledge.txt beside the notebook

In [ ]:
@tool
def automotive_testing_qa(question: str) -> str:
    """Answer automotive testing questions about CAN, CANoe, CAPL, UDS, HIL, ADAS, pytest and Jenkins."""
    return automotive_qa(question)

automotive_tools = [automotive_testing_qa, ruff_qa]
automotive_agent = LabAgent(automotive_tools)

In [ ]:
# Test 1: only automotive knowledge
answer_automotive = automotive_agent.invoke({"input": "What is HIL testing and which tools can be used for it?"})
print(answer_automotive["output"])


In [ ]:
# Test 2: only Ruff knowledge
answer_ruff = automotive_agent.invoke({"input": "Why might developers choose Ruff over Flake8?"})
print(answer_ruff["output"])


In [ ]:
# Test 3: multi-hop, requiring BOTH knowledge bases
answer_both = automotive_agent.invoke({"input": "How can Ruff help an automotive team that uses Python pytest and Jenkins for regression testing? Explain what the automotive testing document says and what Ruff documentation says."})
print(answer_both["output"])


In [ ]:
# Deterministic single-source router: no deprecated return_direct or initialize_agent.
automotive_router = LabAgent(automotive_tools, direct=True)
router_answer = automotive_router.invoke({"input": "What is UDS in automotive testing?"})
print(router_answer["output"])


### Reflection (expected behavior; verify against your own verbose logs)

- **Automotive question:** The agent should select `Automotive_Testing_QA`, which contains the HIL and UDS definitions.
- **Ruff question:** The agent should select `Ruff_Documentation_QA` for the Ruff/Flake8 comparison.
- **Multi-hop question:** With `return_direct=False`, the agent can call both tools and synthesize a combined answer. Actual tool choices depend on the model and retrieval results.
- **Router mode:** With `return_direct=True`, the first tool result is returned immediately; this is faster for single-source questions but can prevent multi-source synthesis.

**Note:** These are expected outcomes, not claims that live API calls were executed. Running the notebook requires an OpenAI API key, internet access for the Ruff documentation, and API usage credits.
